# Chapter 1 &mdash; Where You Will Meet This Again

**Concept 17 of the Chapter 1 decomposition:** *Why This Material Matters for Lifelong Learning*

Hidden Markov Models, deep packet inspection, HTML parsing &mdash; all DFA underneath. Plus what it cost the field to <i>not</i> have this theory.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter1/Concept-Why-This-Matters/Concept-Why-This-Matters.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, sys

try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    ! if [ -d Jove ]; then git -C Jove pull -q --ff-only; else git clone -q https://github.com/ganeshutah/Jove Jove; fi
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
sys.path.insert(0, JOVE)

from jove.LangDef        import *
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.AnimateDFA     import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


* **Machine learning**: a Hidden Markov Model is essentially a DFA whose transitions
  carry probabilities. Speech recognition and phone word-completion are HMMs.
* **Networking**: deep packet inspection scans bit-streams for malware patterns &mdash; DFA,
  and they must be *fast*, or attackers simply flood them.
* **Documents**: handling HTML means handling complex syntax-spaces correctly.

And the cost of not knowing: before precedence was folded into grammars, compilers
replaced each `+` with `)))+(((`. A Venus probe was allegedly lost to a period typed
for a comma.

## 2. Definitions

### Deep packet inspection: a DFA that hunts a signature

Scan a stream for the pattern `1101` &mdash; a stand-in for a malware signature. The DFA
never backs up, so it runs in one pass.

In [ ]:
dpi = md2mc('''DFA
I    : 1 -> S1
I    : 0 -> I
S1   : 1 -> S11
S1   : 0 -> I
S11  : 1 -> S11
S11  : 0 -> S110
S110 : 1 -> F
S110 : 0 -> I
F    : 0 | 1 -> F
''')
print("DPI DFA states :", sorted(dpi["Q"]))

### An HMM is a DFA with probabilities

Same shape, weighted edges. If you understand the DFA, you understand the skeleton.

In [ ]:
hmm = {                       # state -> symbol -> (next_state, probability)
    'Calm':  {'q': ('Calm', 0.8), 'l': ('Loud', 0.2)},
    'Loud':  {'q': ('Calm', 0.4), 'l': ('Loud', 0.6)},
}

def hmm_path_prob(obs, start='Calm'):
    st, p = start, 1.0
    for o in obs:
        st, step = hmm[st][o]
        p *= step
    return st, p

## 3. Tests

The signature scanner, in one pass over the stream.

In [ ]:
for pkt in ['0001101000', '1111', '110', '1101', '0011010']:
    print("%-12s contains 1101? %s" % (pkt, accepts_dfa(dpi, pkt)))

Speed matters for a security reason: a scanner too slow to keep up is a
**denial-of-service** target.

In [ ]:
import time
# NOTE: Jove's accepts_dfa recurses once per input symbol, so Python's
# recursion limit (not the DFA!) caps how long a stream we may feed it.
# The DFA itself would happily run forever in constant memory.
stream = ('0110' * 150) + '1101'
t0 = time.time(); hit = accepts_dfa(dpi, stream); dt = time.time() - t0
print("scanned %d symbols in %.5fs -> found signature: %s" % (len(stream), dt, hit))
assert hit
print("one pass, constant MACHINE memory -- that is why DFA are used here")
print()
print("(The recursion limit is an artefact of this implementation,")
print(" not of the automaton. A loop-based runner scans megabytes.)")

The HMM, walking the same kind of graph with probabilities attached.

In [ ]:
for obs in ['qqq', 'qll', 'lll']:
    st, p = hmm_path_prob(obs)
    print("observations %-5s -> end state %-5s probability %.4f" % (obs, st, p))
print()
print("Strip the numbers and you have a DFA. That is why DFA come first.")

## 4. Animation


Watch the signature scanner. Notice it **never backs up** &mdash; on a mismatch it falls
back to the right partial state and keeps going. That is what makes one pass enough.


*(The `display(HTML(...))` line loads the toolbar's font-awesome icons. Keep it last in the cell &mdash; it must be there for the controls to appear.)*

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(dpi, FuseEdges=True)
display(HTML('<link rel="stylesheet" href="//stackpath.bootstrapcdn.com/font-awesome/4.7.0/css/font-awesome.min.css"/>'))

## 5. Exercises


1. Change the signature from `1101` to `1011`. Which fall-back edges change?
2. Add a third state to the HMM and re-run `hmm_path_prob`. Do the probabilities
   along a path still multiply?
3. Look up the `DO 5 K=1.3` Fortran story. What single property of the *grammar*
   made that typo catastrophic rather than a syntax error?

In [ ]:
# Your work for the exercises above.